# Writes and Python triggers

Calculate and validate stored totals with Python hooks, compare normal writes with direct SQL, and undo changes in the same in-memory database.

[Watch lesson 8](https://asqueel.org/learn/06-aggregations/)

**Run the cells from top to bottom in a fresh Python kernel.** Each notebook includes its own starting state. SQLite runs in memory; restarting the kernel discards the database. To repeat the exercise, restart the kernel and run all cells. Colab execution requires a Google account.

© 2026 Softwell S.r.l. · Apache-2.0

## Before the examples: `_ = ...` is optional

It assigns a return value we do not need, preventing its automatic display in the Python console or notebook. The call still runs normally. This is a Python convention, not an Asqueel requirement. When you need the result, use a meaningful variable name.

## Prepare the notebook

The video starts in a terminal. Here, install the same official package in the notebook kernel. No virtual environment or separate Python console is needed in Colab.

In [ ]:
%pip install -q "asqueel[migration]==0.6.0"

In [ ]:
%config InteractiveShell.ast_node_interactivity = "all"

## Recreate the starting point

The previous lessons are rebuilt below. The four fictional CSV files are included in this notebook and written into a new temporary folder. No downloads or existing database are needed.

In [ ]:
#@title Prepare the sample CSV files
from pathlib import Path
import os
import tempfile
_lesson_folder = tempfile.TemporaryDirectory(prefix="asqueel-notebook-")
os.chdir(_lesson_folder.name)
_csv_files = {'customers.csv': 'id,name,city\n1,Ada,Florence\n2,Grace,London\n3,Linus,Helsinki\n4,Guido,Amsterdam\n5,Elena,Florence\n6,Alan,Manchester\n7,Margaret,New York\n8,Barbara,Boston\n', 'invoice_rows.csv': 'id,invoice_id,product_code,quantity,unit_price,total\n1,101,TEA,2,3.50,7.00\n2,101,COF,1,6.90,6.90\n3,101,MUG,4,5.25,21.00\n4,102,BTEA,2,3.20,6.40\n5,102,HTEA,3,4.10,12.30\n6,102,HON,4,7.50,30.00\n7,103,SUG,3,2.30,6.90\n8,103,CHO,4,3.90,15.60\n9,103,BIS,1,2.80,2.80\n10,104,JAM,4,4.60,18.40\n11,104,OAT,1,2.40,2.40\n12,104,JUI,2,2.70,5.40\n13,105,WAT,1,0.90,0.90\n14,105,LEM,2,1.80,3.60\n15,105,KET,3,32.90,98.70\n16,106,POT,2,18.50,37.00\n17,106,PLT,3,6.40,19.20\n18,106,BWL,4,7.80,31.20\n19,107,SPO,3,9.50,28.50\n20,107,TRY,4,14.90,59.60\n21,107,JAR,1,8.20,8.20\n22,108,NAP,4,12.00,48.00\n23,108,TWL,1,5.60,5.60\n24,108,FLT,2,2.10,4.20\n25,109,TIN,1,6.70,6.70\n26,109,MAT,2,11.40,22.80\n27,109,CAF,3,29.90,89.70\n28,110,BOT,2,13.50,27.00\n29,110,STR,3,4.80,14.40\n30,110,SCL,4,24.90,99.60\n31,111,TEA,3,3.50,10.50\n32,111,COF,4,6.90,27.60\n33,111,MUG,1,5.25,5.25\n34,112,BTEA,4,3.20,12.80\n35,112,HTEA,1,4.10,4.10\n36,112,HON,2,7.50,15.00\n', 'invoices.csv': 'id,number,customer_id,invoice_date\n101,INV-001,1,2026-09-28\n102,INV-002,2,2026-09-29\n103,INV-003,3,2026-09-30\n104,INV-004,4,2026-10-01\n105,INV-005,5,2026-10-02\n106,INV-006,6,2026-10-03\n107,INV-007,7,2026-10-04\n108,INV-008,8,2026-10-05\n109,INV-009,1,2026-10-06\n110,INV-010,2,2026-10-07\n111,INV-011,5,2026-10-08\n112,INV-012,3,2026-10-09\n', 'products.csv': 'code,name,category,stock\nBTEA,Black tea,Drinks,30\nHTEA,Herbal tea,Drinks,18\nHON,Honey,Food,22\nSUG,Cane sugar,Food,40\nCHO,Dark chocolate,Food,35\nBIS,Biscuits,Food,28\nJAM,Strawberry jam,Food,16\nOAT,Oat drink,Drinks,32\nJUI,Apple juice,Drinks,26\nWAT,Mineral water,Drinks,60\nLEM,Lemonade,Drinks,30\nKET,Kettle,Kitchen,9\nPOT,Teapot,Kitchen,14\nPLT,Plate,Kitchen,36\nBWL,Bowl,Kitchen,24\nSPO,Spoon set,Kitchen,20\nTRY,Serving tray,Kitchen,11\nJAR,Storage jar,Kitchen,25\nNAP,Cotton napkins,Kitchen,18\nTWL,Tea towel,Kitchen,30\nFLT,Coffee filters,Kitchen,45\nTIN,Tea tin,Kitchen,15\nMAT,Table mats,Kitchen,16\nCAF,Coffee maker,Kitchen,7\nBOT,Water bottle,Kitchen,21\nSTR,Tea strainer,Kitchen,23\nSCL,Kitchen scale,Kitchen,8\n'}
for _name, _text in _csv_files.items():
    _ = Path(_name).write_text(_text, encoding="utf-8")


In [ ]:
#@title Recreate the previous lessons

######## Import our two tools ########

from asqueel import AsqueelDb, SqlDatabaseConfig


######## Choose SQLite in memory ########

model = SqlDatabaseConfig()
root = model.source.db(implementation="sqlite")
_ = root.connection(name=":memory:", implementation="sqlite")


######## Describe a product table ########

shop = root.packages().package("shop", sqlschema="shop")
product = shop.table("product", pkey="code")
_ = product.column("code", size=":10")
_ = product.column("name", size=":60")


######## Create that table ########

db = AsqueelDb(model)
print(db.migrate().commands)


######## Add our first product ########

products = db.table("shop.product")
products.insert({"code": "TEA", "name": "Tea"})
db.commit()


######## Read our product ########

products.query(columns="$code, $name").fetch()


######## Change its name ########

products.update({"name": "Green tea"}, pkey="TEA")
db.commit()


######## Check the update ########

products.query(columns="$code, $name").fetch()
db.commit()


######## Track category and stock ########

_ = product.column("category", size=":30")
_ = product.column("stock", dtype="I")


######## Load the updated model ########

db.reloadModel(model)


######## Add those database columns ########

print(db.migrate().commands)


######## Check our original row ########

products = db.table("shop.product")
products.query(columns="$name, $stock").fetch()
db.commit()


######## Give tea its stock ########

products.update({"category": "Drinks", "stock": 8}, pkey="TEA")
db.commit()


######## Read the new values ########

products.query(columns="$name, $category, $stock").fetch()
db.commit()


######## Add products from CSV ########

from io import StringIO
data = StringIO(
    "code,name,category,stock\n"
    "COF,Coffee,Drinks,24\n"
    "MUG,Mug,Kitchen,12\n"
)


######## Import those rows ########

products.sql_insert(data, format="csv")
db.commit()


######## List just the drinks ########

products.query(
    columns="$name, $stock",
    where="$category = :category", category="Drinks",
    order_by="$stock DESC",
).fetch()
db.commit()

# Continue with lesson two.

######## Check our starting point ########

products.query(columns="$code, $name").fetch()
db.commit()


######## Describe our customers ########

customer = shop.table("customer", pkey="id")
_ = customer.column("id", dtype="I")
_ = customer.column("name", size=":60")
_ = customer.column("city", size=":60")


######## Create the customer table ########

db.reloadModel(model)
print(db.migrate().commands)


######## Describe a dated invoice ########

invoice = shop.table("invoice", pkey="id")
_ = invoice.column("id", dtype="I")
_ = invoice.column("number", size=":20")
_ = invoice.column("invoice_date", dtype="D")


######## Connect invoices to customers ########

customer_id = invoice.column("customer_id", dtype="I")
_ = customer_id.relation(
    "shop.customer.id", foreign_key=True, relation_name="invoices",
)


######## Describe an invoice row ########

invoice_row = shop.table("invoice_row", pkey="id")
_ = invoice_row.column("id", dtype="I")
_ = invoice_row.column("quantity", dtype="I")
_ = invoice_row.column("unit_price", dtype="N", size="12,2")
_ = invoice_row.column("total", dtype="N", size="12,2")


######## Connect the row at both ends ########

invoice_id = invoice_row.column("invoice_id", dtype="I")
_ = invoice_id.relation(
    "shop.invoice.id", foreign_key=True, relation_name="rows",
)
product_code = invoice_row.column("product_code", size=":10")
_ = product_code.relation(
    "shop.product.code", foreign_key=True, relation_name="rows",
)


######## Apply the new model ########

db.reloadModel(model)
print(db.migrate().commands)


######## Get fresh table handles ########

products = db.table("shop.product")
customers = db.table("shop.customer")
invoices = db.table("shop.invoice")
rows = db.table("shop.invoice_row")


######## Inspect the row file ########

from pathlib import Path
print("\n".join(Path("invoice_rows.csv").read_text().splitlines()[:4]))


######## Expand the product catalogue ########

with open("products.csv", newline="", encoding="utf-8") as source:
    products.sql_insert(source)

db.commit()


######## Count the catalogue ########

products.query().count()
db.commit()


######## Import customers and their cities ########

with open("customers.csv", newline="", encoding="utf-8") as source:
    customers.sql_insert(source)

db.commit()
customers.query(columns="$name, $city", limit=3).fetch()
db.commit()


######## Import the invoices ########

with open("invoices.csv", newline="", encoding="utf-8") as source:
    invoices.sql_insert(source)

db.commit()


######## Import the invoice rows ########

with open("invoice_rows.csv", newline="", encoding="utf-8") as source:
    rows.sql_insert(source)

db.commit()


######## Check how much we imported ########

invoices.query().count()
rows.query().count()
db.commit()


######## Read invoices with their customers ########

invoices.query(
    columns="$number, $invoice_date, "
            "@customer_id.name, @customer_id.city",
    limit=3,
).fetch()
db.commit()


######## Inspect the first invoice ########

rows.query(
    columns="@product_code.name, "
            "$quantity, $unit_price, $total",
    where="$invoice_id = :invoice", invoice=101,
).fetch()
db.commit()


######## Read across both connections ########

rows.query(
    columns="@invoice_id.number, "
            "@invoice_id.@customer_id.city, "
            "@product_code.name, $total",
    limit=3,
).fetch()
db.commit()

# Continue with lesson three.

######## Find invoices for Florence ########

invoices.query(
    columns="$number",
    where="@customer_id.city = :city", city="Florence",
).fetch()


######## Sort invoices by city ########

invoices.query(
    columns="$number, @customer_id.city",
    order_by="@customer_id.city", limit=4,
).fetch()


######## Show the newest rows first ########

rows.query(
    columns="@invoice_id.invoice_date, $total",
    order_by="@invoice_id.invoice_date DESC", limit=3,
).fetch()


######## Find rows for Florence ########

rows.query(
    columns="@product_code.name, $quantity",
    where="@invoice_id.@customer_id.city = :city",
    city="Florence", limit=3,
).fetch()


######## Combine the conditions ########

from datetime import date
rows.query(
    columns="@product_code.name, $total",
    where="@invoice_id.@customer_id.city = :city "
          "AND @invoice_id.invoice_date >= :since",
    city="Florence", since=date(2026, 10, 1),
    order_by="@invoice_id.invoice_date DESC", limit=4,
).fetch()


######## Close this read transaction ########

db.commit()

# Continue with lesson 4.

######## Call this path city ########

_ = invoice_row.aliasColumn(
    "city", relation_path="@invoice_id.@customer_id.city",
)
db.reloadModel(model)
rows = db.table("shop.invoice_row")


######## Read and filter with that name ########

rows.query(
    columns="$city, @product_code.name",
    where="$city = :city", city="Florence", limit=3,
).fetch()
db.commit()


######## Calculate quantity times price ########

_ = invoice_row.formulaColumn(
    "calculated_total",
    sql_formula="ROUND($quantity * $unit_price, 2)",
    dtype="N",
)
db.reloadModel(model)
rows = db.table("shop.invoice_row")


######## Read the calculated amounts ########

rows.query(
    columns="$quantity, $unit_price, $calculated_total",
    where="$invoice_id = :invoice", invoice=101,
    order_by="$id",
).fetch()
db.commit()


######## Check whether the database needs changing ########

db.migration_plan().empty


######## Temporarily change the quantity to three ########

rows.update({"quantity": 3}, pkey=1)
rows.query(
    columns="$quantity, $total, $calculated_total",
    where="$id = :id", id=1,
).fetch()


######## Undo that change ########

db.rollback()

# Continue with lesson 5.

######## Give each invoice a calculated total ########

_ = invoice.formulaColumn(
    "total", dtype="N",
    select=dict(
        table="shop.invoice_row",
        columns="ROUND(SUM($calculated_total), 2)",
        where="$invoice_id = #THIS.id",
    ),
)
db.reloadModel(model)
invoices = db.table("shop.invoice")


######## Read those totals as ordinary fields ########

invoices.query(
    columns="$number, $total", order_by="$id", limit=3,
).fetch()
db.commit()


######## Total the invoices of each customer ########

_ = customer.formulaColumn(
    "total", dtype="N",
    select=dict(
        table="shop.invoice",
        columns="ROUND(SUM($total), 2)",
        where="$customer_id = #THIS.id",
    ),
)
db.reloadModel(model)
customers = db.table("shop.customer")


######## Read each customer's sales total ########

customers.query(
    columns="$name, $total", order_by="$id", limit=3,
).fetch()
db.commit()


######## Total the sales of each product ########

_ = product.formulaColumn(
    "total", dtype="N",
    select=dict(
        table="shop.invoice_row",
        columns="ROUND(SUM($calculated_total), 2)",
        where="$product_code = #THIS.code",
    ),
)
db.reloadModel(model)
products = db.table("shop.product")
rows = db.table("shop.invoice_row")


######## Read the product totals ########

products.query(
    columns="$code, $name, $total",
    where="$code IN :codes", codes=["TEA", "COF", "MUG"],
    order_by="$code",
).fetch()
db.commit()


######## Summarize each customer's invoices ########

customers = db.table("shop.customer")
customers.query(
    columns="$name, @invoices.aggr("
            "sum($total), count(), avg($total))",
    order_by="$id", limit=3,
).fetch()
db.commit()


######## Count the rows of each invoice too ########

invoices = db.table("shop.invoice")
invoices.query(
    columns="$number, $total, @rows.aggr(count())",
    order_by="$id", limit=3,
).fetch()
db.commit()


######## Average the recorded selling prices ########

products.query(
    columns="$code, @rows.aggr(count(), avg($unit_price))",
    where="$code IN :codes", codes=["TEA", "COF", "MUG"],
    order_by="$code",
).fetch()
db.commit()


######## Name the average price so we can filter it ########

_ = product.formulaColumn(
    "average_price", dtype="N",
    select=dict(
        table="shop.invoice_row", columns="AVG($unit_price)",
        where="$product_code = #THIS.code",
    ),
)
db.reloadModel(model)
products = db.table("shop.product")
rows = db.table("shop.invoice_row")


######## Filter by average price ########

products.query(
    columns="$code, $name, $average_price",
    where="$average_price BETWEEN :min_price AND :max_price",
    min_price=3.2, max_price=4.8,
    order_by="$average_price, $code",
).fetch()
db.commit()


## 1 — What we'll learn in this lesson

Welcome back! Here's what we'll learn. Calculate stored row totals when writing, reject invalid quantities, and compare normal writes with direct SQL. These are Python hooks in Asqueel, not SQL triggers installed in SQLite. We keep our database in memory.

```text
What we'll learn in this lesson
Python triggers → calculate and validate writes
insert and update → run the write cycle
sql_insert and sql_update → skip Python triggers
rollback → undo the demonstration

Same SQLite database in memory
```

## 2 — Give invoice rows their Python behavior

Let's define the write behavior. Our shared calculation checks quantity and price, then stores their product rounded to cents. The inserting and updating hooks run before the SQL write. The updated hook prints the old and new quantities afterwards. Afterwards means after the statement, not after commit. For this example, both write methods receive quantity and price.

In [ ]:
######## Define the write behavior ########

from asqueel import SqlTable
from decimal import Decimal

class InvoiceRowTable(SqlTable):
    def calculate_total(self, record):
        if record["quantity"] <= 0:
            raise ValueError("Quantity must be positive")
        price = Decimal(str(record["unit_price"]))
        if price < 0:
            raise ValueError("Price must not be negative")
        record["total"] = (
            record["quantity"] * price
        ).quantize(Decimal("0.01"))
    def trigger_onInserting(self, record):
        self.calculate_total(record)
    def trigger_onUpdating(self, record, old_record=None):
        self.calculate_total(record)
    def trigger_onUpdated(self, record, old_record=None):
        if old_record is not None:
            print("Quantity:", old_record["quantity"],
                  "->", record["quantity"])

Let's attach it to invoice rows. X table class gives this existing table its Python behavior. Reload the model and get a fresh table handle. True means no migration is needed: we added Python hooks, not columns. Existing rows are not recalculated.

In [ ]:
######## Attach it to invoice rows ########

_ = shop.table("invoice_row", x_table_class=InvoiceRowTable)
db.reloadModel(model)
rows = db.table("shop.invoice_row")
db.migration_plan().empty

## 3 — Insert a row without supplying its total

Let's insert two teas. We supply quantity and price, but no total. The inserting hook stores seven. The formula column from lesson four also reads seven. The stored total is written by our hook; the formula is calculated when we query. This change is still uncommitted.

In [ ]:
######## Insert two teas ########

_ = rows.insert({
    "id": 901, "invoice_id": 101, "product_code": "TEA",
    "quantity": 2, "unit_price": Decimal("3.50"),
})
rows.query(
    columns="$id, $quantity, $unit_price, $total, $calculated_total",
    where="$id = :id", id=901,
).fetchRecord()

## 4 — Update a record with its previous values

Let's change the quantity to three. We read the physical fields and copy them before editing. Passing old record gives the hooks the previous values; table update does not fetch or merge them for us. The updating hook recalculates ten fifty. The updated hook prints two to three. Its print is immediate, even though we have not committed.

In [ ]:
######## Change the quantity to three ########

old = dict(rows.query(where="$id = :id", id=901).fetchRecord())
changed = dict(old, quantity=3)
_ = rows.update(changed, old_record=old)
rows.query(
    columns="$quantity, $total, $calculated_total",
    where="$id = :id", id=901,
).fetchRecord()

## 5 — Direct SQL skips the Python hooks

Let's compare a direct SQL update. SQL update changes one selected row and returns one. It skips our Python hooks: no message appears, and the stored total stays ten fifty. The formula now reads fourteen. Database constraints still apply, but Python calculations and validation are our responsibility on this path.

In [ ]:
######## Compare a direct SQL update ########

rows.query(where="$id = :id", id=901).sql_update({"quantity": 4})
rows.query(
    columns="$quantity, $total, $calculated_total",
    where="$id = :id", id=901,
).fetchRecord()

## 6 — Update a selection through the write cycle

Let's run the normal write cycle again. Query update reads each selected record, merges our changes and runs table update with the old record. Unlike the direct table call, it prepares the complete records for us. The hooks run, both totals become seventeen fifty, and the returned list contains the updated primary key.

In [ ]:
######## Run the normal write cycle again ########

rows.query(where="$id = :id", id=901).update({"quantity": 5})
rows.query(
    columns="$quantity, $total, $calculated_total",
    where="$id = :id", id=901,
).fetchRecord()

## 7 — Direct insertion follows the same rule

Let's compare a direct insertion. SQL insert stores the supplied ninety-nine unchanged. The formula reads seven. The direct CSV import from lesson two uses this same SQL path, so its source file must already contain correct totals. Use normal insert when you want our Python hooks to calculate them.

In [ ]:
######## Compare a direct insertion ########

_ = rows.sql_insert({
    "id": 902, "invoice_id": 101, "product_code": "TEA",
    "quantity": 2, "unit_price": Decimal("3.50"),
    "total": Decimal("99.00"),
})
rows.query(
    columns="$quantity, $total, $calculated_total",
    where="$id = :id", id=902,
).fetchRecord()

## 8 — Undo the uncommitted examples

Let's undo our demonstration writes. Rollback removes both new rows and their updates together, returning us to thirty-six original rows. It does not erase messages already printed by a Python hook. Commit here closes the verification read; it does not save the rolled-back examples.

In [ ]:
######## Undo our demonstration writes ########

db.rollback()
rows.query(where="$id >= :first_demo", first_demo=901).fetch()
rows.query().count()
db.commit()

## 9 — Reject an invalid write and recover

Let's reject a zero quantity. Our inserting hook raises a value error before SQL insertion. After a failed write cycle, roll back before continuing. That rollback affects all pending work in the transaction, not just this row. We began from a clean boundary, so no earlier work is lost. The empty result confirms that row nine hundred and three was never stored.

In [ ]:
######## Reject a zero quantity ########

try:
    rows.insert({
        "id": 903, "invoice_id": 101, "product_code": "TEA",
        "quantity": 0, "unit_price": Decimal("3.50"),
    })
except ValueError as error:
    print(str(error))
    db.rollback()

rows.query(where="$id = :id", id=903).fetch()
db.commit()

## 10 — What to remember

Thanks for following along! Use normal writes for Python validation and calculations. Use direct SQL deliberately, with complete valid data. Keep transaction boundaries explicit. We have now taken the same model from reads and reports to its write behavior.

```text
Thank you!
insert / update → Python hooks + SQL
query(...).update(...) → write cycle for each selected row
sql_insert / query(...).sql_update(...) → direct SQL
After a statement ≠ after commit
rollback → all pending database changes

Continue with the course reading lessons
```

## Check your result

Run this final cell to check the exercise.

In [ ]:
from math import isclose
original_rows = [{'id': '1', 'invoice_id': '101', 'product_code': 'TEA', 'quantity': '2', 'unit_price': '3.50', 'total': '7.00'}, {'id': '2', 'invoice_id': '101', 'product_code': 'COF', 'quantity': '1', 'unit_price': '6.90', 'total': '6.90'}, {'id': '3', 'invoice_id': '101', 'product_code': 'MUG', 'quantity': '4', 'unit_price': '5.25', 'total': '21.00'}, {'id': '4', 'invoice_id': '102', 'product_code': 'BTEA', 'quantity': '2', 'unit_price': '3.20', 'total': '6.40'}, {'id': '5', 'invoice_id': '102', 'product_code': 'HTEA', 'quantity': '3', 'unit_price': '4.10', 'total': '12.30'}, {'id': '6', 'invoice_id': '102', 'product_code': 'HON', 'quantity': '4', 'unit_price': '7.50', 'total': '30.00'}, {'id': '7', 'invoice_id': '103', 'product_code': 'SUG', 'quantity': '3', 'unit_price': '2.30', 'total': '6.90'}, {'id': '8', 'invoice_id': '103', 'product_code': 'CHO', 'quantity': '4', 'unit_price': '3.90', 'total': '15.60'}, {'id': '9', 'invoice_id': '103', 'product_code': 'BIS', 'quantity': '1', 'unit_price': '2.80', 'total': '2.80'}, {'id': '10', 'invoice_id': '104', 'product_code': 'JAM', 'quantity': '4', 'unit_price': '4.60', 'total': '18.40'}, {'id': '11', 'invoice_id': '104', 'product_code': 'OAT', 'quantity': '1', 'unit_price': '2.40', 'total': '2.40'}, {'id': '12', 'invoice_id': '104', 'product_code': 'JUI', 'quantity': '2', 'unit_price': '2.70', 'total': '5.40'}, {'id': '13', 'invoice_id': '105', 'product_code': 'WAT', 'quantity': '1', 'unit_price': '0.90', 'total': '0.90'}, {'id': '14', 'invoice_id': '105', 'product_code': 'LEM', 'quantity': '2', 'unit_price': '1.80', 'total': '3.60'}, {'id': '15', 'invoice_id': '105', 'product_code': 'KET', 'quantity': '3', 'unit_price': '32.90', 'total': '98.70'}, {'id': '16', 'invoice_id': '106', 'product_code': 'POT', 'quantity': '2', 'unit_price': '18.50', 'total': '37.00'}, {'id': '17', 'invoice_id': '106', 'product_code': 'PLT', 'quantity': '3', 'unit_price': '6.40', 'total': '19.20'}, {'id': '18', 'invoice_id': '106', 'product_code': 'BWL', 'quantity': '4', 'unit_price': '7.80', 'total': '31.20'}, {'id': '19', 'invoice_id': '107', 'product_code': 'SPO', 'quantity': '3', 'unit_price': '9.50', 'total': '28.50'}, {'id': '20', 'invoice_id': '107', 'product_code': 'TRY', 'quantity': '4', 'unit_price': '14.90', 'total': '59.60'}, {'id': '21', 'invoice_id': '107', 'product_code': 'JAR', 'quantity': '1', 'unit_price': '8.20', 'total': '8.20'}, {'id': '22', 'invoice_id': '108', 'product_code': 'NAP', 'quantity': '4', 'unit_price': '12.00', 'total': '48.00'}, {'id': '23', 'invoice_id': '108', 'product_code': 'TWL', 'quantity': '1', 'unit_price': '5.60', 'total': '5.60'}, {'id': '24', 'invoice_id': '108', 'product_code': 'FLT', 'quantity': '2', 'unit_price': '2.10', 'total': '4.20'}, {'id': '25', 'invoice_id': '109', 'product_code': 'TIN', 'quantity': '1', 'unit_price': '6.70', 'total': '6.70'}, {'id': '26', 'invoice_id': '109', 'product_code': 'MAT', 'quantity': '2', 'unit_price': '11.40', 'total': '22.80'}, {'id': '27', 'invoice_id': '109', 'product_code': 'CAF', 'quantity': '3', 'unit_price': '29.90', 'total': '89.70'}, {'id': '28', 'invoice_id': '110', 'product_code': 'BOT', 'quantity': '2', 'unit_price': '13.50', 'total': '27.00'}, {'id': '29', 'invoice_id': '110', 'product_code': 'STR', 'quantity': '3', 'unit_price': '4.80', 'total': '14.40'}, {'id': '30', 'invoice_id': '110', 'product_code': 'SCL', 'quantity': '4', 'unit_price': '24.90', 'total': '99.60'}, {'id': '31', 'invoice_id': '111', 'product_code': 'TEA', 'quantity': '3', 'unit_price': '3.50', 'total': '10.50'}, {'id': '32', 'invoice_id': '111', 'product_code': 'COF', 'quantity': '4', 'unit_price': '6.90', 'total': '27.60'}, {'id': '33', 'invoice_id': '111', 'product_code': 'MUG', 'quantity': '1', 'unit_price': '5.25', 'total': '5.25'}, {'id': '34', 'invoice_id': '112', 'product_code': 'BTEA', 'quantity': '4', 'unit_price': '3.20', 'total': '12.80'}, {'id': '35', 'invoice_id': '112', 'product_code': 'HTEA', 'quantity': '1', 'unit_price': '4.10', 'total': '4.10'}, {'id': '36', 'invoice_id': '112', 'product_code': 'HON', 'quantity': '2', 'unit_price': '7.50', 'total': '15.00'}]
actual = rows.query(columns="$id,$quantity,$unit_price,$total", order_by="$id").fetch()
assert len(actual) == len(original_rows) == 36
for actual_row, source in zip(actual, sorted(original_rows, key=lambda r: int(r["id"]))):
    assert actual_row["id"] == int(source["id"])
    assert actual_row["quantity"] == int(source["quantity"])
    assert isclose(float(actual_row["unit_price"]), float(source["unit_price"]))
    assert isclose(float(actual_row["total"]), float(source["total"]))

assert isinstance(rows, InvoiceRowTable)
assert db.table("shop.product").query().count() == 30
assert db.table("shop.customer").query().count() == 8
assert db.table("shop.invoice").query().count() == 12
db.commit()
rows.insert(dict(id=999, invoice_id=101, product_code="TEA", quantity=2, unit_price=Decimal("3.50")))
assert rows.query(where="$id=999").fetchRecord()["total"] == 7
assert rows.query(where="$id=999").update({"quantity": 3}) == [999]
assert rows.query(where="$id=999").fetchRecord()["total"] == Decimal("10.50")
assert rows.query(where="$id=999").sql_update({"quantity": 4}) == 1
assert rows.query(where="$id=999").fetchRecord()["total"] == Decimal("10.50")
db.rollback()
assert rows.query().count() == 36
db.commit()

db.commit()
db.commit()
print("Lesson checks passed.")
